In [ ]:
# PURPOSE OF NOTEBOOK

# Data cleaning
# Final Standardization
# Derived metrics
# Feature engineering
# Joins
# Data quality validation
# Analytics-ready modeling

##### Advanced operations>>
# Advanced STEP 1 Repartitioned 
# Advanced STEP 2 Broadcast Match Join
# Advanced  STEP 3 — Cache Silver Dataset
# Advanced STEP 4 — Repartition Before Heavy Aggregation
# Advanced  STEP 5 — Coalesce Before CSV Export

# 1. Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


import warnings
warnings.filterwarnings("ignore")

# Section 2 — Spark Session

In [2]:

try:
    spark.stop()
except:
    pass
from pyspark.sql import SparkSession 

spark = (
    SparkSession.builder
    .appName("S3App")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 07:37:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/26 07:37:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/26 07:37:25 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
spark

# 2. Read data from Read Bronze Layer parquet files

In [4]:
# # 2.1 path variables for data files writing path_to_parquet

# ball_by_ball_file_path_to_parquet = "/opt/spark-data/ipl_data/2_bronze/ball_by_ball_parquet"
# match_file_path_to_parquet= "/opt/spark-data/ipl_data/2_bronze/match_parquet"
# player_file_path_to_parquet= "/opt/spark-data/ipl_data/2_bronze/player_parquet"
# player_match_file_path_to_parquet= "/opt/spark-data/ipl_data/2_bronze/player_match_parquet"
# team_file_path_to_parquet= "/opt/spark-data/ipl_data/2_bronze/team_parquet"


# 2.1 path variables for data files writing path_to_parquet
# ==========================
# Bronze Layer
# ==========================
BRONZE_PATH = "/opt/spark-data/2_bronze"

ball_by_ball_file_path_to_parquet = f"{BRONZE_PATH}/ball_by_ball_parquet"
match_file_path_to_parquet = f"{BRONZE_PATH}/match_parquet"
player_file_path_to_parquet = f"{BRONZE_PATH}/player_parquet"
player_match_file_path_to_parquet = f"{BRONZE_PATH}/player_match_parquet"
team_file_path_to_parquet = f"{BRONZE_PATH}/team_parquet"

In [5]:
# 2.2 Read Bronze Layer parquet files

ball_by_ball_df = spark.read.parquet(ball_by_ball_file_path_to_parquet)

match_df = spark.read.parquet(match_file_path_to_parquet)

player_df = spark.read.parquet(player_file_path_to_parquet)

player_match_df = spark.read.parquet(player_match_file_path_to_parquet)

team_df = spark.read.parquet(team_file_path_to_parquet)

In [6]:
# 2.3 Checking columns of data read from Bronze
# print(ball_by_ball_df.columns)
# print(match_df.columns)
print(player_df.columns)
# print(player_match_df.columns)
# print(team_df.columns)

['player_sk', 'player_id', 'player_name', 'dob', 'batting_hand', 'bowling_skill', 'country_name']


In [7]:
player_df.select('player_id', 'player_name').show()

+---------+----------------+
|player_id|     player_name|
+---------+----------------+
|      152|        A Mukund|
|      444|       MB Parmar|
|       74|        GC Smith|
|      340|       GH Vihari|
|      382|       YS Chahal|
|      339|         KK Nair|
|      493|     T Natarajan|
|      256|   Harmeet Singh|
|      388|         S Gopal|
|      442|       RA Shaikh|
|      232|        UT Yadav|
|       64|DPMD Jayawardene|
|      124|        A Kumble|
|      170|   Yashpal Singh|
|      221|      KA Pollard|
|      246|        AN Ahmed|
|      365|        UA Birla|
|      456|      K Santokie|
|      190|        M Morkel|
|      463|        T Shamsi|
+---------+----------------+
only showing top 20 rows



# 3. ball_by_ball TABLE Repartition

In [8]:
# 1]“Repartitioned large fact dataset before heavy transformations.”
# improve parallelism
# balanced processing
# better joins/aggregations

ball_by_ball_df = ball_by_ball_df.repartition(8)


# 3.1 ball_by_ball TABLE TRANSFORMATIONS

In [9]:
# 3.1 Handle Null Numeric Columns
numeric_cols = [
    "runs_scored",
    "extra_runs",
    "wides",
    "legbyes",
    "byes",
    "noballs",
    "penalty",
    "bowler_extras"
]

ball_by_ball_df = ball_by_ball_df.fillna(0, subset=numeric_cols)

In [10]:
# 3.2 Handle Null Boolean Columns
bool_cols = [
    "caught",
    "bowled",
    "run_out",
    "lbw",
    "retired_hurt",
    "stumped",
    "caught_and_bowled",
    "hit_wicket",
    "obstructingfeild",
    "bowler_wicket",
    "keeper_catch"
]

ball_by_ball_df = ball_by_ball_df.fillna(False, subset=bool_cols)

In [11]:
# 3.3 Standardize Team Names
ball_by_ball_df = (
    ball_by_ball_df
    .withColumn("team_batting", trim(initcap(col("team_batting"))))
    .withColumn("team_bowling", trim(initcap(col("team_bowling"))))
)

In [12]:
# 3.4 Create Total Runs Column
ball_by_ball_df = ball_by_ball_df.withColumn(
    "total_runs",
    col("runs_scored") + col("extra_runs")
)

In [13]:
# 3.5 Create Boundary Flags
ball_by_ball_df = (
    ball_by_ball_df
    .withColumn(
        "is_four",
        when(col("runs_scored") == 4, 1).otherwise(0)
    )
    .withColumn(
        "is_six",
        when(col("runs_scored") == 6, 1).otherwise(0)
    )
)

In [14]:
# 3.6 Create Dot Ball Flag
ball_by_ball_df = ball_by_ball_df.withColumn(
    "is_dot_ball",
    when(col("total_runs") == 0, 1).otherwise(0)
)

In [15]:
# 3.7 Create Wicket Flag
ball_by_ball_df = ball_by_ball_df.withColumn(
    "is_wicket",
    when(col("player_out").isNotNull(), 1).otherwise(0)
)

In [16]:
# 3.8 Over + Ball Unique Identifier
ball_by_ball_df = ball_by_ball_df.withColumn(
    "over_ball",
    concat_ws(".", col("over_id"), col("ball_id"))
)

In [17]:
# 3.9 Create Powerplay / Middle / Death Overs
ball_by_ball_df = (
    ball_by_ball_df
    .withColumn(
        "innings_phase",
        when(col("over_id") <= 6, "Powerplay")
        .when((col("over_id") > 6) & (col("over_id") <= 15), "Middle")
        .otherwise("Death")
    )
)

In [18]:
# 3.10 Strike Rotation Indicator
ball_by_ball_df = ball_by_ball_df.withColumn(
    "strike_rotation",
    when(col("runs_scored").isin([1,3]), 1).otherwise(0)
)

# 4. MATCH TABLE TRANSFORMATIONS

In [19]:
match_df.select(
    "match_id",
    "match_date"
).show(5, False)

+--------+----------+
|match_id|match_date|
+--------+----------+
|501214  |2011-04-15|
|548319  |NULL      |
|729306  |2014-04-25|
|829784  |NULL      |
|980952  |2016-04-28|
+--------+----------+
only showing top 5 rows



In [20]:
match_df.select("match_date").show(5)

+----------+
|match_date|
+----------+
|2011-04-15|
|      NULL|
|2014-04-25|
|      NULL|
|2016-04-28|
+----------+
only showing top 5 rows



In [21]:
# 4.1 Standardize Team Names
team_cols = [
    "team1",
    "team2",
    "toss_winner",
    "match_winner"
]

for c in team_cols:
    match_df = match_df.withColumn(
        c,
        trim(initcap(col(c)))
    )

In [22]:
# 4.2 Venue Cleaning
match_df = match_df.withColumn(
    "venue_name",
    trim(col("venue_name"))
)

In [23]:
# 4.3 Create Match Result Column
match_df = match_df.withColumn(
    "match_result",
    concat_ws(
        " ",
        col("match_winner"),
        lit("won by"),
        col("win_margin"),
        col("win_type")
    )
)


In [24]:
# 4.4 Toss Winner Advantage
match_df = match_df.withColumn(
    "toss_winner_match_winner_same",
    when(col("toss_winner") == col("match_winner"), 1).otherwise(0)
)


In [25]:
match_df

DataFrame[match_sk: int, match_id: int, team1: string, team2: string, match_date: date, season_year: int, venue_name: string, city_name: string, country_name: string, toss_winner: string, match_winner: string, toss_name: string, win_type: string, outcome_type: string, manofmach: string, win_margin: int, country_id: int, match_result: string, toss_winner_match_winner_same: int]

In [26]:
# 4.5 Season Label
match_df = match_df.withColumn(
    "season",
    concat(lit("IPL "), col("season_year"))
)

# 5. PLAYER TABLE TRANSFORMATIONS

In [27]:
# 5.1 Clean Names
player_df = player_df.withColumn(
    "player_name",
    trim(initcap(col("player_name")))
)

In [28]:
# 5.2 Age Calculation
player_df = player_df.withColumn(
    "player_age",
    floor(months_between(current_date(), col("dob")) / 12)
)

In [29]:
# 5.3 Standardize Batting/Bowling Style
player_df = (
    player_df
    .withColumn("batting_hand", trim(initcap(col("batting_hand"))))
    .withColumn("bowling_skill", trim(initcap(col("bowling_skill"))))
)

# 6. PLAYER MATCH TRANSFORMATIONS

In [30]:
# 6.1 Standardize Team Names
player_match_df = (
    player_match_df
    .withColumn("player_team", trim(initcap(col("player_team"))))
    .withColumn("opposit_team", trim(initcap(col("opposit_team"))))
)

In [31]:
# 6.2 Role Cleaning
player_match_df = player_match_df.withColumn(
    "role_desc",
    trim(initcap(col("role_desc")))
)


In [32]:
# 6.3 Create Experience Bucket
player_match_df = player_match_df.withColumn(
    "age_bucket",
    when(col("age_as_on_match") < 25, "Young")
    .when((col("age_as_on_match") >= 25) & (col("age_as_on_match") < 33), "Prime")
    .otherwise("Senior")
)

In [33]:
# 6.4 Man of Match Flag
player_match_df = player_match_df.withColumn(
    "mom_flag",
    when(col("is_manofthematch") == True, 1).otherwise(0)
)

# 7. TEAM TABLE TRANSFORMATIONS

In [34]:
# 7.1 Team Name Cleaning
team_df = team_df.withColumn(
    "team_name",
    trim(initcap(col("team_name")))
)

# 8. SILVER joins__ Broadcast JOINS (IMPORTANT)

Your final_silver_df is basically:

ball_by_ball_df
    + match_df
    + player_df

joined together into one analytical dataframe.
# ==========================================

This table becomes:

batting analytics base
bowling analytics base
strike rate calculations
economy calculations




analytics-ready dataset
fact-style table


WHY HERE?

Because:

match_df

is small.

Broadcast sends small table to all executors instead of expensive shuffle.

Huge Spark optimization.

In [35]:
8.1 # Detect Duplicate Columns
common_cols = list(
    set(ball_by_ball_df.columns)
    .intersection(set(match_df.columns))
)

print(common_cols)

['match_date', 'season', 'match_id']


In [36]:
# 8.2 Create Clean Match Columns

match_cols_to_select = [
    c for c in match_df.columns
    if c not in common_cols
]


print(match_cols_to_select)

['match_sk', 'team1', 'team2', 'season_year', 'venue_name', 'city_name', 'country_name', 'toss_winner', 'match_winner', 'toss_name', 'win_type', 'outcome_type', 'manofmach', 'win_margin', 'country_id', 'match_result', 'toss_winner_match_winner_same']


In [37]:
# 8.1 BALL + MATCH JOIN

# from pyspark.sql.functions import broadcast, col
ball_match_df = (
    ball_by_ball_df.alias("b")
    .join(
        broadcast(match_df.alias("m")),
        col("b.match_id") == col("m.match_id"),
        "left"
    )
    .select(
        "b.*",
        *[col(f"m.{c}") for c in match_cols_to_select]
    )
)



In [38]:
# # 8.2 BALL + PLAYER JOIN (STRIKER)


ball_match_player_df = (
    ball_match_df.alias("bm")
    .join(
        broadcast(player_df.alias("p")),
        col("bm.striker") == col("p.player_id"),
        "left"
    )
    .select(
        "bm.*",   # keep all previous columns
        
        # select only needed player columns
        col("p.player_name").alias("striker_name"),
        col("p.batting_hand"),
        col("p.bowling_skill"),
        col('p.player_id')
    )
)


In [39]:
# ============================================
# BALL + PLAYER JOIN (BOWLER)
# ============================================

final_silver_df = (

    ball_match_player_df.alias("bm")

    .join(
        broadcast(player_df.alias("p")),

        col("bm.bowler") == col("p.player_id"),

        "left"
    )

    .select(

        "bm.*",

        col("p.player_name").alias("bowler_name")

    )
)

In [40]:
final_silver_df.select(
    "bowler",
    "bowler_name"
).show(5, truncate=False)

[Stage 11:>                                                       (0 + 16) / 16]

+------+-----------+
|bowler|bowler_name|
+------+-----------+
|80    |Ds Kulkarni|
|83    |M Kartik   |
|409   |Mp Stoinis |
|357   |Mg Johnson |
|190   |M Morkel   |
+------+-----------+
only showing top 5 rows



In [41]:
final_silver_df.columns

['match_id',
 'over_id',
 'ball_id',
 'innings_no',
 'team_batting',
 'team_bowling',
 'striker_batting_position',
 'extra_type',
 'runs_scored',
 'extra_runs',
 'wides',
 'legbyes',
 'byes',
 'noballs',
 'penalty',
 'bowler_extras',
 'out_type',
 'caught',
 'bowled',
 'run_out',
 'lbw',
 'retired_hurt',
 'stumped',
 'caught_and_bowled',
 'hit_wicket',
 'obstructingfeild',
 'bowler_wicket',
 'match_date',
 'season',
 'striker',
 'non_striker',
 'bowler',
 'player_out',
 'fielders',
 'striker_match_sk',
 'strikersk',
 'nonstriker_match_sk',
 'nonstriker_sk',
 'fielder_match_sk',
 'fielder_sk',
 'bowler_match_sk',
 'bowler_sk',
 'playerout_match_sk',
 'battingteam_sk',
 'bowlingteam_sk',
 'keeper_catch',
 'player_out_sk',
 'matchdatesk',
 'total_runs',
 'is_four',
 'is_six',
 'is_dot_ball',
 'is_wicket',
 'over_ball',
 'innings_phase',
 'strike_rotation',
 'match_sk',
 'team1',
 'team2',
 'season_year',
 'venue_name',
 'city_name',
 'country_name',
 'toss_winner',
 'match_winner',
 'toss

In [42]:
final_silver_df.select(
    "bowler",
    "bowler_name"
).show(5, truncate=False)

+------+-----------+
|bowler|bowler_name|
+------+-----------+
|80    |Ds Kulkarni|
|83    |M Kartik   |
|409   |Mp Stoinis |
|357   |Mg Johnson |
|190   |M Morkel   |
+------+-----------+
only showing top 5 rows



# 9. DATA QUALITY CHECKS

In [43]:
# 9.1 Duplicate Check
duplicate_count = (
    ball_by_ball_df.groupBy(
        "match_id",
        "innings_no",
        "over_id",
        "ball_id"
    )
    .count()
    .filter(col("count") > 1)
)

duplicate_count.show()

[Stage 22:=============================>                            (4 + 4) / 8]

+--------+----------+-------+-------+-----+
|match_id|innings_no|over_id|ball_id|count|
+--------+----------+-------+-------+-----+
+--------+----------+-------+-------+-----+



In [44]:
# to check duplicate columns exist
print(len(final_silver_df.columns))
print(len(set(final_silver_df.columns)))

78
78


In [45]:
# 9.2 Null Check
final_silver_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in final_silver_df.columns
]).show()

26/06/26 07:37:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+--------+-------+-------+----------+------------+------------+------------------------+----------+-----------+----------+-----+-------+----+-------+-------+-------------+--------+------+------+-------+---+------------+-------+-----------------+----------+----------------+-------------+----------+------+-------+-----------+------+----------+--------+----------------+---------+-------------------+-------------+----------------+----------+---------------+---------+------------------+--------------+--------------+------------+-------------+-----------+----------+-------+------+-----------+---------+---------+-------------+---------------+--------+-----+-----+-----------+----------+---------+------------+-----------+------------+---------+--------+------------+---------+----------+----------+------------+-----------------------------+------------+------------+-------------+---------+-----------+
|match_id|over_id|ball_id|innings_no|team_batting|team_bowling|striker_batting_position|extra_t

In [46]:
# match df found null date column work on it

match_df.select(
    "match_id",
    "match_date"
).show(2, False)

+--------+----------+
|match_id|match_date|
+--------+----------+
|501214  |2011-04-15|
|548319  |NULL      |
+--------+----------+
only showing top 2 rows



In [47]:
final_silver_df.printSchema()

root
 |-- match_id: integer (nullable = true)
 |-- over_id: integer (nullable = true)
 |-- ball_id: integer (nullable = true)
 |-- innings_no: integer (nullable = true)
 |-- team_batting: string (nullable = true)
 |-- team_bowling: string (nullable = true)
 |-- striker_batting_position: integer (nullable = true)
 |-- extra_type: string (nullable = true)
 |-- runs_scored: integer (nullable = true)
 |-- extra_runs: integer (nullable = true)
 |-- wides: integer (nullable = true)
 |-- legbyes: integer (nullable = true)
 |-- byes: integer (nullable = true)
 |-- noballs: integer (nullable = true)
 |-- penalty: integer (nullable = true)
 |-- bowler_extras: integer (nullable = true)
 |-- out_type: string (nullable = true)
 |-- caught: boolean (nullable = false)
 |-- bowled: boolean (nullable = false)
 |-- run_out: boolean (nullable = false)
 |-- lbw: boolean (nullable = false)
 |-- retired_hurt: boolean (nullable = false)
 |-- stumped: boolean (nullable = false)
 |-- caught_and_bowled: boolean

In [48]:
final_silver_df.select(
    "match_id",
    "match_date"
).show(2, False)

+--------+----------+
|match_id|match_date|
+--------+----------+
|419168  |2010-04-22|
|734030  |2014-05-21|
+--------+----------+
only showing top 2 rows



# 10. WRITE SILVER TABLES

In [49]:
# # path variables for data files writing path_to_parquet to 3_silver
# final_silver_df_file_path_to_parquet="/opt/spark-data/ipl_data/3_silver/final_silver_parquet"

# ball_by_ball_file_path_to_parquet = "/opt/spark-data/ipl_data/3_silver/ball_by_ball_parquet"
# match_file_path_to_parquet= "/opt/spark-data/ipl_data/3_silver/match_parquet"
# player_file_path_to_parquet= "/opt/spark-data/ipl_data/3_silver/player_parquet"
# player_match_file_path_to_parquet= "/opt/spark-data/ipl_data/3_silver/player_match_parquet"
# team_file_path_to_parquet= "/opt/spark-data/ipl_data/3_silver/team_parquet"


# Base path for Silver layer
SILVER_PATH = "/opt/spark-data/3_silver"

# Final Silver DataFrame
final_silver_df_file_path_to_parquet = f"{SILVER_PATH}/final_silver_parquet"

# Silver parquet paths
ball_by_ball_file_path_to_parquet = f"{SILVER_PATH}/ball_by_ball_parquet"
match_file_path_to_parquet = f"{SILVER_PATH}/match_parquet"
player_file_path_to_parquet = f"{SILVER_PATH}/player_parquet"
player_match_file_path_to_parquet = f"{SILVER_PATH}/player_match_parquet"
team_file_path_to_parquet = f"{SILVER_PATH}/team_parquet"

In [50]:
# 10.1 Save Ball silver Table

ball_by_ball_df.write \
    .mode("overwrite") \
    .parquet(ball_by_ball_file_path_to_parquet)

In [51]:
# 10.2 Save Match Table

# Write to silver parquet Dataset
match_df.write.mode("overwrite").parquet(match_file_path_to_parquet)

In [52]:
# 10.3 Save Player Table

# Write to silver parquet Dataset
player_df.write \
    .mode("overwrite") \
    .parquet(player_file_path_to_parquet)

In [53]:
# 10.4 Save Player Match Table


# Write to silver parquet Dataset
player_match_df.write \
    .mode("overwrite") \
    .parquet(player_match_file_path_to_parquet)

In [54]:
# 10.5 Save Final Joined Silver Dataset

# Write to silver parquet Dataset
team_df.write \
    .mode("overwrite") \
    .parquet(team_file_path_to_parquet)

# . SILVER joins__ Broadcast JOINS (IMPORTANT)

Your final_silver_df is basically:

ball_by_ball_df
    + match_df
    + player_df

joined together into one analytical dataframe.
# ==========================================

This table becomes:

batting analytics base,
bowling analytics base,
strike rate calculations,
economy calculations,




analytics-ready dataset
fact-style table


WHY HERE?

Because:

match_df

is small.

Broadcast sends small table to all executors instead of expensive shuffle.

Huge Spark optimization.

In [55]:
# Final Joined Silver Dataset
# Your final_silver_df is basically:
# ball_by_ball_df
#     + match_df
#     + player_df
# joined together into one analytical dataframe.



final_silver_df.write \
    .mode("overwrite") \
    .parquet(final_silver_df_file_path_to_parquet)

In [56]:
ls

01_ipl_data_ingestion.ipynb
02_ipl_data_validate_write_bronze.ipynb*
03_ipl_data_Transform_silver.ipynb*
04_ipl_data_Transformation_gold.ipynb*
05_spark_sql_analytic.ipynb
